# Section 2 · Notebook 2 — Prebuilt Tools and AWS Integration (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file. Parts 1–2 need nothing else. Part 3 drives AWS itself (S3, DynamoDB, Lambda), so it still needs AWS credentials — but no Bedrock access.

1. The prebuilt calculator tool
2. Combining several prebuilt tools in one request
3. Talking to AWS in plain English with `use_aws` — S3, DynamoDB and Lambda

> Part 3 creates a few small AWS resources (an S3 bucket, a DynamoDB table, a Lambda function and its IAM role). The last cell deletes all of them.

In [1]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.


d:\work\AI Bootcamp\self_practice\myvenv\Scripts\python.exe: No module named pip


In [2]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
# Part 3 still talks to AWS (S3, DynamoDB, Lambda), so it needs a Region and AWS credentials.
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq (AWS Region for part 3: {REGION})")

Using openai/gpt-oss-120b on Groq (AWS Region for part 3: us-east-1)


---
## 1. Calculator Tool

Expressions, equations and calculus — all from a single import.

In [3]:
from strands import Agent
from strands_tools import calculator

agent = Agent(model=model, tools=[calculator], system_prompt="You are a helpful math assistant.")

agent("What's 42 raised to the power of 9?")
agent("Solve the equation x^2 + 5x + 6 = 0")
agent("What's the derivative of sin(x) * cos(x)?")

DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 42**9               │                                                                            │
│  │ Result    │ 406671383849472     │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The value of \(42^9\) is  

\[
\boxed{406\,671\,383\,849\,472}.
\]To solve  

\[
x^{2}+5x+6=0,
\]

factor the quadratic:

\[
x^{2}+5x+6=(x+2)(x+3)=0.
\]

Set each factor to zero:

\[
\begin{cases}
x+2 = 0 \;\Rightarrow\; x = -2,\\[4pt]
x+3 = 0 \;\Rightarrow\; x = -3.
\end{cases}
\]

\[
\boxed{x = -2 \text{ or } x = -3}
\]To differentiate  

\[
f(x)=\sin x \,\cos x,
\]

apply the product rule:

\[
\frac{d}{dx}\bigl[\sin x \,\cos x\bigr]
   = (\sin x)'\,\cos x + \sin x\,(\cos x)' .
\]

Since  

\[
(\sin x)' = \cos x \quad\text{and}\quad (\cos x)' = -\sin x,
\]

we get  

\[
\frac{d}{dx}\bigl[\sin x \,\cos x\bigr]
   = \cos x \,\cos x + \sin x\,(-\sin x)
   = \cos^{2}x - \sin^{2}x .
\]

Using the double‑angle identity \(\cos 2x = \cos^{2}x - \sin^{2}x\), the derivative can also be written as  

\[
\boxed{\displaystyle \frac{d}{dx}\bigl[\sin x \,\cos x\bigr]=\cos^{2}x - \sin^{2}x = \cos 2x }.
\]

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "To differentiate  \n\n\\[\nf(x)=\\sin x \\,\\cos x,\n\\]\n\napply the product rule:\n\n\\[\n\\frac{d}{dx}\\bigl[\\sin x \\,\\cos x\\bigr]\n   = (\\sin x)'\\,\\cos x + \\sin x\\,(\\cos x)' .\n\\]\n\nSince  \n\n\\[\n(\\sin x)' = \\cos x \\quad\\text{and}\\quad (\\cos x)' = -\\sin x,\n\\]\n\nwe get  \n\n\\[\n\\frac{d}{dx}\\bigl[\\sin x \\,\\cos x\\bigr]\n   = \\cos x \\,\\cos x + \\sin x\\,(-\\sin x)\n   = \\cos^{2}x - \\sin^{2}x .\n\\]\n\nUsing the double‑angle identity \\(\\cos 2x = \\cos^{2}x - \\sin^{2}x\\), the derivative can also be written as  \n\n\\[\n\\boxed{\\displaystyle \\frac{d}{dx}\\bigl[\\sin x \\,\\cos x\\bigr]=\\cos^{2}x - \\sin^{2}x = \\cos 2x }.\n\\]"}], 'metadata': {'usage': {'inputTokens': 1536, 'outputTokens': 307, 'totalTokens': 1843}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 944}}, 'tracking_id': '72367e55-908b-4976-8f5f-fc273ac6e771'}, metrics=EventLoopMetrics(cycle_cou

---
## 2. Combining Multiple Tools

Fetch live data, crunch the numbers, and save the result — one request, three tools.

`file_write` normally asks you to confirm before writing a file. `BYPASS_TOOL_CONSENT=true` skips that prompt for this demo; leave it off in real applications.

In [4]:
from strands_tools import file_write, http_request

os.environ["BYPASS_TOOL_CONSENT"] = "true"

agent = Agent(
    model=model,
    tools=[http_request, calculator, file_write],
    system_prompt="You help with data analysis tasks.",
)

agent("""
Fetch the 7-day forecast from
https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature_2m_max,temperature_2m_min&timezone=Asia/Kolkata
extract the daily maximum temperatures for Bangalore,
calculate the average maximum temperature over the period,
and save a short summary to weather_summary.txt
""")

DEPRECATION WARNING: http_request is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the http_request and web_fetch tools vended by strands-agents (from strands.vended_tools import http_request, web_fetch). http_request handles API calls; web_fetch retrieves and reads web pages.



Tool #1: http_request


╭─────────────────────────────────── 🚀 HTTP Request Preview: GET /v1/forecast ───────────────────────────────────╮
│                                                                                                                 │
│   Method    GET                                                                                                 │
│   URL       https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature_2m_max,t…   │
│   Headers   {}                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sending request...

╭─────────────────────────────────────────── ✅ HTTP Response: 200 OK ────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│     Status         200 OK                                                                                       │
│     URL            https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature…     │
│     Content-Type   application/json; charset=utf-8                                                              │
│     Size           503 bytes                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                     Response Headers                      
╭───────────────────────┬─────────────────────────────────╮
│ Header                │ Value                           │
├───────────────────────┼─────────────────────────────────┤
│ Date                  │ Wed, 04 Nov 2026 06:19:58 GMT   │
│ Content-Type          │ application/json; charset=utf-8 │
│ Transfer-Encoding     │ chunked                         │
│ Connection            │ keep-alive                      │
│ X-RateLimit-Cost      │ 1.0                             │
│ X-RateLimit-Limit     │ 600.0                           │
│ X-RateLimit-Remaining │ 599.0                           │
│ X-RateLimit-Reset     │ 1793773258                      │
│ X-RateLimit-Window    │ 60                              │
│ Content-Encoding      │ deflate                         │
╰───────────────────────┴─────────────────────────────────╯


Tool #2: file_write


╔══ File Write Operation ═══╗
║                           ║
║ Path: weather_summary.txt ║
║ Size: 178 characters      ║
║                           ║
╚═══════════════════════════╝

╔════════════════ Write Successful ════════════════╗
║ File written successfully to weather_summary.txt ║
╚══════════════════════════════════════════════════╝

The 7‑day forecast for Bangalore (2026‑11‑04 to 2026‑11‑10) shows daily maximum temperatures of **27.7 °C, 28.1 °C, 28.2 °C, 28.2 °C, 27.1 °C, 27.0 °C, and 27.7 °C**.  

The average maximum temperature over the period is **≈ 27.71 °C**.

A summary has been saved to **`weather_summary.txt`**.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The 7‑day forecast for Bangalore (2026‑11‑04\u202fto\u202f2026‑11‑10) shows daily maximum temperatures of **27.7\u202f°C, 28.1\u202f°C, 28.2\u202f°C, 28.2\u202f°C, 27.1\u202f°C, 27.0\u202f°C, and 27.7\u202f°C**.  \n\nThe average maximum temperature over the period is **≈\u202f27.71\u202f°C**.\n\nA summary has been saved to **`weather_summary.txt`**.'}], 'metadata': {'usage': {'inputTokens': 2559, 'outputTokens': 118, 'totalTokens': 2677}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 7043}}, 'tracking_id': '03e15cfb-85ce-43ab-ad54-61d5a5920f8e'}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'http_request': ToolMetrics(tool={'toolUseId': 'fc_4acb45c6-bd28-489e-9541-34c3926d1bff', 'name': 'http_request', 'input': {'allow_redirects': True, 'method': 'GET', 'timeout': 30, 'url': 'https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature_2m_max,temperature_2m_min

In [ ]:
if os.path.exists("weather_summary.txt"):
    print(open("weather_summary.txt").read())
else:
    print("No weather_summary.txt yet — scroll up to see which tools the agent called.")

---
## 3. AWS Integration with `use_aws`

`use_aws` turns plain English into AWS API calls. It uses the same credentials as your notebook, so it can only do what your IAM permissions allow.

### 3.0 Create the demo resources

This cell creates:
- an S3 bucket with a `sales-2024.csv` file
- a DynamoDB table `customers-<suffix>` with customer `12345`
- a Lambda function `order-processor-<suffix>` (plus an IAM role for it)

A random suffix keeps the names unique, so nothing you already have is touched.

In [5]:
import csv
import io
import json
import time
import zipfile

import boto3

session = boto3.Session(region_name=REGION)
account_id = session.client("sts").get_caller_identity()["Account"]
s3 = session.client("s3")
dynamodb = session.client("dynamodb")
iam = session.client("iam")
lambda_client = session.client("lambda")

import uuid

suffix = uuid.uuid4().hex[:6]  # unique names, so we never touch resources you already have
BUCKET = f"agent-tools-demo-{account_id}-{REGION}-{suffix}"
TABLE = f"customers-{suffix}"
FUNCTION = f"order-processor-{suffix}"
ROLE_NAME = f"order-processor-demo-role-{suffix}"

# --- S3 bucket + CSV -------------------------------------------------------
if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET)
else:
    s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})

rows = [
    ("month", "region", "revenue"),
    ("2024-01", "North", 120000), ("2024-02", "North", 135000), ("2024-03", "North", 142000),
    ("2024-01", "South", 98000), ("2024-02", "South", 104000), ("2024-03", "South", 111000),
]
buf = io.StringIO()
csv.writer(buf).writerows(rows)
s3.put_object(Bucket=BUCKET, Key="sales-2024.csv", Body=buf.getvalue().encode())
print(f"S3 bucket ready: s3://{BUCKET}/sales-2024.csv")

# --- DynamoDB table + one customer -----------------------------------------
dynamodb.create_table(
    TableName=TABLE,
    KeySchema=[{"AttributeName": "customer_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "customer_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
)
dynamodb.get_waiter("table_exists").wait(TableName=TABLE)
dynamodb.put_item(
    TableName=TABLE,
    Item={"customer_id": {"S": "12345"}, "name": {"S": "Asha Rao"}, "email": {"S": "asha@example.com"}},
)
print(f"DynamoDB table ready: {TABLE}")

# --- Lambda function ---------------------------------------------------------
trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
role_arn = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
time.sleep(10)  # let IAM propagate

handler_code = '''
import json

def handler(event, context):
    order_id = event.get("order_id", "unknown")
    return {"order_id": order_id, "status": "PROCESSED", "message": f"Order {order_id} processed successfully"}
'''
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w") as z:
    z.writestr("lambda_function.py", handler_code)

lambda_client.create_function(
    FunctionName=FUNCTION,
    Runtime="python3.12",
    Role=role_arn,
    Handler="lambda_function.handler",
    Code={"ZipFile": zip_buf.getvalue()},
    Timeout=10,
)
lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION)
print(f"Lambda function ready: {FUNCTION}")

S3 bucket ready: s3://agent-tools-demo-815398003681-us-east-1-63d08e/sales-2024.csv
DynamoDB table ready: customers-63d08e
Lambda function ready: order-processor-63d08e


### 3.1 List your S3 buckets

> **Groq note:** `use_aws` lets the model pick an AWS profile with `profile_name`, and `gpt-oss` models like to send `profile_name="default"`, which overrides an `AWS_PROFILE` you set (an SSO profile, for example). So we hand the agent a copy of the tool with that option removed: every call uses this notebook's credentials. It is the same idea as Section 3, where the memory tool's user is fixed in code — decide in code what the model must not choose.

In [6]:
import copy

from strands.tools.tools import PythonAgentTool
from strands_tools import use_aws as use_aws_module

# use_aws without the profile_name option (see the note above)
spec = copy.deepcopy(use_aws_module.TOOL_SPEC)
del spec["inputSchema"]["json"]["properties"]["profile_name"]
use_aws = PythonAgentTool("use_aws", spec, use_aws_module.use_aws)

aws_agent = Agent(
    model=model,
    tools=[use_aws, calculator],
    system_prompt=f"You are an AWS assistant that helps manage cloud resources. Use region {REGION}.",
)

aws_agent("List all S3 buckets in my account")


Tool #1: use_aws


╭───── 🚀 List S3 buckets ──────╮
│                               │
│  Service:       s3            │
│  Operation:     list_buckets  │
│  Region:        us-east-1     │
│  Parameters:    None          │
│                               │
╰───────────────────────────────╯

Here are the S3 buckets in your account (region **us‑east‑1**):

| Bucket Name | Creation Date (UTC) | ARN |
|-------------|---------------------|-----|
| **agent-tools-demo-815398003681-us-east-1-63d08e** | 2026‑11‑04 06:24:08 | `arn:aws:s3:::agent-tools-demo-815398003681-us-east-1-63d08e` |

If you need more details or want to perform actions on this bucket (e.g., list objects, delete, apply policies), just let me know!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here are the S3 buckets in your account (region **us‑east‑1**):\n\n| Bucket Name | Creation Date (UTC) | ARN |\n|-------------|---------------------|-----|\n| **agent-tools-demo-815398003681-us-east-1-63d08e** | 2026‑11‑04\u202f06:24:08 | `arn:aws:s3:::agent-tools-demo-815398003681-us-east-1-63d08e` |\n\nIf you need more details or want to perform actions on this bucket (e.g., list objects, delete, apply policies), just let me know!'}], 'metadata': {'usage': {'inputTokens': 1859, 'outputTokens': 134, 'totalTokens': 1993}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 611}}, 'tracking_id': 'cd835d0e-fa44-4c3d-97bb-f08b853be1cf'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'use_aws': ToolMetrics(tool={'toolUseId': 'fc_d709ef05-ec86-4c4c-91f3-63a2e4f43998', 'name': 'use_aws', 'input': {'label': 'List S3 buckets', 'operation_name': 'list_buckets', 'parameters': {}, 'region': 'us-east-1', '

### 3.2 Data analysis on S3

Read a CSV from S3, total the revenue, and write a summary back to S3.

In [7]:
# Fresh conversation per task: Groq's free tier rejects any single request over 8,000 tokens
aws_agent.messages.clear()

aws_agent(
    f"Read the sales data from S3 bucket '{BUCKET}' key 'sales-2024.csv', "
    "calculate the total revenue, and save a short summary back to the same bucket as 'sales-summary.txt'"
)


Tool #2: use_aws


╭─────────────────── 🚀 Get sales CSV from S3 ────────────────────╮
│                                                                 │
│  Service:       s3                                              │
│  Operation:     get_object                                      │
│  Region:        us-east-1                                       │
│  Parameters:                                                    │
│    • Bucket:    agent-tools-demo-815398003681-us-east-1-63d08e  │
│    • Key:       sales-2024.csv                                  │
│                                                                 │
╰─────────────────────────────────────────────────────────────────╯


Tool #3: use_aws


╭─────────────────── 🚀 Upload sales summary to S3 ────────────────────╮
│                                                                      │
│  Service:       s3                                                   │
│  Operation:     put_object                                           │
│  Region:        us-east-1                                            │
│  Parameters:                                                         │
│    • Body:      Total revenue for 2024 across all regions: $710,000  │
│    • Bucket:    agent-tools-demo-815398003681-us-east-1-63d08e       │
│    • Key:       sales-summary.txt                                    │
│                                                                      │
╰──────────────────────────────────────────────────────────────────────╯

The sales data has been processed:

**Total revenue for 2024 across all regions:** **$710,000**

A summary file (`sales-summary.txt`) containing this information has been saved back to the S3 bucket `agent-tools-demo-815398003681-us-east-1-63d08e`.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The sales data has been processed:\n\n**Total revenue for 2024 across all regions:** **$710,000**\n\nA summary file (`sales-summary.txt`) containing this information has been saved back to the S3 bucket `agent-tools-demo-815398003681-us-east-1-63d08e`.'}], 'metadata': {'usage': {'inputTokens': 2508, 'outputTokens': 67, 'totalTokens': 2575}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 569}}, 'tracking_id': '9fa060ca-e4e4-4397-86b6-1082c4d12d6e'}, metrics=EventLoopMetrics(cycle_count=5, tool_metrics={'use_aws': ToolMetrics(tool={'toolUseId': 'fc_598a54f3-ffa9-4f26-852c-24eb6a8b943c', 'name': 'use_aws', 'input': {'label': 'Upload sales summary to S3', 'operation_name': 'put_object', 'parameters': {'Body': 'Total revenue for 2024 across all regions: $710,000', 'Bucket': 'agent-tools-demo-815398003681-us-east-1-63d08e', 'Key': 'sales-summary.txt'}, 'region': 'us-east-1', 'service_name': 's3'}}, call

### 3.3 Customer service on DynamoDB

In [8]:
aws_agent.messages.clear()  # fresh conversation per task (see 3.2)

aws_agent(
    f"Look up customer ID 12345 in the DynamoDB table '{TABLE}' and update their email to newemail@example.com. "
    "Then read the item back to confirm."
)


Tool #4: use_aws


╭───────────────── 🚀 Update email for customer ID 12345 ──────────────────╮
│                                                                          │
│  Service:                         dynamodb                               │
│  Operation:                       update_item                            │
│  Region:                          us-east-1                              │
│  Parameters:                                                             │
│    • ExpressionAttributeValues:   {':e': {'S': 'newemail@example.com'}}  │
│    • Key:                         {'customer_id': {'N': '12345'}}        │
│    • ReturnValues:                ALL_NEW                                │
│    • TableName:                   customers-63d08e                       │
│    • UpdateExpression:            SET email = :e                         │
│                                                                          │
╰──────────────────────────────────────────────────────────────────────────╯

AWS call threw exception: ClientError



Tool #5: use_aws


╭──── 🚀 Describe customers table ────╮
│                                     │
│  Service:         dynamodb          │
│  Operation:       describe_table    │
│  Region:          us-east-1         │
│  Parameters:                        │
│    • TableName:   customers-63d08e  │
│                                     │
╰─────────────────────────────────────╯


Tool #6: use_aws


╭─────────── 🚀 Update email for customer ID 12345 (string key) ───────────╮
│                                                                          │
│  Service:                         dynamodb                               │
│  Operation:                       update_item                            │
│  Region:                          us-east-1                              │
│  Parameters:                                                             │
│    • ExpressionAttributeValues:   {':e': {'S': 'newemail@example.com'}}  │
│    • Key:                         {'customer_id': {'S': '12345'}}        │
│    • ReturnValues:                ALL_NEW                                │
│    • TableName:                   customers-63d08e                       │
│    • UpdateExpression:            SET email = :e                         │
│                                                                          │
╰──────────────────────────────────────────────────────────────────────────╯


Tool #7: use_aws


╭─────────── 🚀 Get updated customer item ───────────╮
│                                                    │
│  Service:         dynamodb                         │
│  Operation:       get_item                         │
│  Region:          us-east-1                        │
│  Parameters:                                       │
│    • Key:         {'customer_id': {'S': '12345'}}  │
│    • TableName:   customers-63d08e                 │
│                                                    │
╰────────────────────────────────────────────────────╯

The customer record was successfully updated and verified.

**Updated item (customer_id = "12345"):**
- `customer_id`: "12345"
- `name`: "Asha Rao"
- `email`: **newemail@example.com**

The email address now reflects the new value as requested.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The customer record was successfully updated and verified.\n\n**Updated item (customer_id\u202f=\u202f"12345"):**\n- `customer_id`: "12345"\n- `name`: "Asha Rao"\n- `email`: **newemail@example.com**\n\nThe email address now reflects the new value as requested.'}], 'metadata': {'usage': {'inputTokens': 2775, 'outputTokens': 66, 'totalTokens': 2841}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 11261}}, 'tracking_id': '1bc5a484-5018-4877-96a9-77e913b45241'}, metrics=EventLoopMetrics(cycle_count=10, tool_metrics={'use_aws': ToolMetrics(tool={'toolUseId': 'fc_383bada7-7631-4a55-864f-62d47fdcbf22', 'name': 'use_aws', 'input': {'label': 'Get updated customer item', 'operation_name': 'get_item', 'parameters': {'Key': {'customer_id': {'S': '12345'}}, 'TableName': 'customers-63d08e'}, 'region': 'us-east-1', 'service_name': 'dynamodb'}}, call_count=7, success_count=6, error_count=1, total_time=11.26600000

### 3.4 Workflow orchestration with Lambda

In [9]:
aws_agent.messages.clear()  # fresh conversation per task (see 3.2)

aws_agent(f"Invoke the Lambda function '{FUNCTION}' with the payload {{\"order_id\": \"67890\"}} and tell me the result")


Tool #8: use_aws


╭─ 🚀 Invoke order-processor-63d08e Lambda with order_id 67890 ─╮
│                                                               │
│  Service:              lambda                                 │
│  Operation:            invoke                                 │
│  Region:               us-east-1                              │
│  Parameters:                                                  │
│    • FunctionName:     order-processor-63d08e                 │
│    • InvocationType:   RequestResponse                        │
│    • Payload:          {"order_id": "67890"}                  │
│                                                               │
╰───────────────────────────────────────────────────────────────╯

The Lambda function **order‑processor‑63d08e** was invoked successfully with the payload `{"order_id":"67890"}`.

**Result**

```json
{
  "order_id": "67890",
  "status": "PROCESSED",
  "message": "Order 67890 processed successfully"
}
```

The order has been processed without errors.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The Lambda function **order‑processor‑63d08e** was invoked successfully with the payload `{"order_id":"67890"}`.\n\n**Result**\n\n```json\n{\n  "order_id": "67890",\n  "status": "PROCESSED",\n  "message": "Order 67890 processed successfully"\n}\n```\n\nThe order has been processed without errors.'}], 'metadata': {'usage': {'inputTokens': 1804, 'outputTokens': 81, 'totalTokens': 1885}, 'metrics': {'latencyMs': 0, 'timeToFirstByteMs': 2964}}, 'tracking_id': '13583126-8b60-44ab-8aae-563ffc593833'}, metrics=EventLoopMetrics(cycle_count=12, tool_metrics={'use_aws': ToolMetrics(tool={'toolUseId': 'fc_89cfff87-0a2f-4f11-89d5-3acc2e34345d', 'name': 'use_aws', 'input': {'label': 'Invoke order-processor-63d08e Lambda with order_id 67890', 'operation_name': 'invoke', 'parameters': {'FunctionName': 'order-processor-63d08e', 'InvocationType': 'RequestResponse', 'Payload': '{"order_id": "67890"}'}, 'region': 'us-

### 3.5 Clean up

Deletes everything created in 3.0.

In [10]:
# S3: delete all objects, then the bucket
for obj in s3.list_objects_v2(Bucket=BUCKET).get("Contents", []):
    s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
s3.delete_bucket(Bucket=BUCKET)
print("Deleted bucket", BUCKET)

dynamodb.delete_table(TableName=TABLE)
print("Deleted table", TABLE)

lambda_client.delete_function(FunctionName=FUNCTION)
iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
iam.delete_role(RoleName=ROLE_NAME)
print("Deleted Lambda function and role")

Deleted bucket agent-tools-demo-815398003681-us-east-1-63d08e
Deleted table customers-63d08e
Deleted Lambda function and role
